In [ ]:
from datetime import datetime as dt

track_data = {
    "datetime": [
        "2014-09-11 00:00", "2014-09-11 06:00", "2014-09-11 12:00",
        "2014-09-11 18:00", "2014-09-12 00:00", "2014-09-12 06:00", "2014-09-12 12:00",
        "2014-09-12 18:00", "2014-09-13 00:00", "2014-09-13 06:00", "2014-09-13 12:00",
        "2014-09-13 18:00", "2014-09-14 00:00", "2014-09-14 06:00", "2014-09-14 12:00",
        "2014-09-14 18:00", "2014-09-15 00:00", "2014-09-15 06:00", "2014-09-15 12:00",
        "2014-09-15 18:00", "2014-09-16 00:00", "2014-09-16 06:00", "2014-09-16 12:00",
        "2014-09-16 18:00", "2014-09-17 00:00", "2014-09-17 06:00", "2014-09-17 12:00",
        "2014-09-17 18:00", "2014-09-18 00:00", "2014-09-18 06:00", "2014-09-18 12:00",
        "2014-09-18 18:00", "2014-09-19 00:00", "2014-09-19 06:00", "2014-09-19 12:00",
        "2014-09-19 18:00", "2014-09-20 00:00", "2014-09-20 06:00", "2014-09-20 12:00",
        "2014-09-20 18:00"
    ],
    "lat": [
        14.2, 14.9, 15.8, 16.5, 17.2, 17.7, 18.3, 18.9, 19.6, 20.4, 21.4, 22.3,
        23.0, 23.7, 24.4, 25.1, 25.8, 26.5, 27.0, 27.7, 28.5, 29.5, 30.6, 31.7,
        32.9, 34.3, 35.7, 37.2, 38.3, 39.3, 39.9, 39.9, 39.9, 39.9, 39.5, 39.5,
        39.4, 39.3, 39.1, 38.8
    ],
    "lon": [
        -34.4, -35.5, -36.5, -37.7, -39.2, -40.8, -42.3, -43.6, -44.8,
        -45.7, -46.4, -47.0, -47.9, -49.0, -50.3, -51.5, -52.7, -54.0, -55.1,
        -56.1, -56.7, -57.3, -57.8, -57.7, -57.0, -56.0, -54.5, -52.4, -49.5,
        -46.7, -44.1, -42.1, -40.1, -38.9, -38.1, -37.4, -36.6, -35.7, -34.4, -33.0
    ],
    "pressure": [
        1009, 1009, 1008, 1007, 1006, 1005, 1000, 999, 998, 997, 996, 995,
        994, 991, 985, 980, 975, 970, 966, 962, 960, 956, 955, 958, 959, 958,
        957, 955, 958, 964, 969, 973, 980, 988, 994, 1001, 1004, 1006, 1008, 1010
    ],
    "wind": [
        25, 25, 30, 30, 35, 35, 40, 40, 40, 45, 45, 50, 55, 60, 70, 75, 80,
        85, 90, 95, 95, 100, 105, 95, 85, 85, 80, 80, 75, 70, 70, 65, 60, 55,
        45, 40, 35, 30, 30, 25
    ],
    "stage": [
        "low", "low", "tropical depression", "tropical depression",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "tropical storm", "tropical storm", "tropical storm", "low", "low",
        "low", "low", "low"
    ]
}


In [ ]:
import pandas as pd
import xarray as xr
import glob
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
import numpy as np
import os
import datetime as dt
import subprocess
from tropycal import tracks

#########################
# Animation time range
#########################
start_cutoff = pd.Timestamp("2014-09-11 00:00:00")
end_cutoff = pd.Timestamp("2014-09-18 23:59:00")

#########################
# Load storm track data
#########################
df_track = pd.DataFrame(track_data)
df_track["datetime"] = pd.to_datetime(df_track["datetime"])
df_track = df_track.set_index("datetime")

# Interpolate numeric columns every 30 min
interp_times = pd.date_range(
    start=df_track.index.min(), end=df_track.index.max(), freq="30min"
)
numeric_cols = ["lat", "lon", "wind", "pressure"]
track_interp = df_track[numeric_cols].reindex(df_track.index.union(interp_times))
track_interp = track_interp.interpolate(method="time")
track_interp = track_interp.loc[
    (track_interp.index >= start_cutoff) & (track_interp.index <= end_cutoff)
]

# Forward-fill stage column
track_interp["stage"] = df_track["stage"].reindex(track_interp.index).ffill()

#########################
# Load RMW data
#########################
rmw_file = "RMW Edouard - Sheet2.csv"
df_rmw = pd.read_csv(rmw_file, skiprows=1)
df_rmw['datetime'] = pd.to_datetime(
    df_rmw['date'] + ' ' + df_rmw['time'], format="%m-%d-%Y %H:%M:%S"
)
df_rmw = df_rmw.set_index('datetime')

# Interpolate RMW to 30-min track times
interp_times = track_interp.index
rmw_interp = np.interp(
    interp_times.astype(int),
    df_rmw.index.astype(int),
    df_rmw['km']
)
df_rmw_interp = pd.DataFrame({'RMW_km': rmw_interp}, index=interp_times)

#########################
# Output folder
#########################
output_dir = "outputs_edouard_frames"
os.makedirs(output_dir, exist_ok=True)

#########################
# Load MERGIR data
#########################
merg_files = sorted(glob.glob("/spare11/atm350/es348865/MERGIR_data/merg_201409*.nc4"))
ds_merg = xr.open_mfdataset(merg_files, combine="by_coords")

#########################
# Load Lightning data
#########################
lightning_files = sorted(
    glob.glob("/nfs/home11/ugrad/2024/es348865/honors_research/250523/WWLLN_201409*.csv")
)
df_light_all = pd.concat([pd.read_csv(f) for f in lightning_files])
df_light_all["longitude"] = ((df_light_all["longitude"] + 180) % 360) - 180
df_light_all["datetime"] = pd.to_datetime(
    df_light_all["datetime"], format="%Y/%m/%d %H:%M:%S.%f"
)

#########################
# Load SHIPS shear data
#########################
basin = tracks.TrackDataset(basin="north_atlantic", include_btk=False)
storm = basin.get_storm(("edouard", 2014))
ships = storm.get_ships(dt.datetime(2014, 9, 11, 12))
valid_times = [ships.attrs["forecast_init"] + dt.timedelta(hours=int(h)) for h in ships.fhr]

df_ships = pd.DataFrame({
    "datetime": valid_times,
    "shear_mag": ships.shear_kt,
    "shear_dir": ships.shear_dir
})
df_ships["u_shear"] = -df_ships["shear_mag"] * np.sin(np.deg2rad(df_ships["shear_dir"]))
df_ships["v_shear"] = -df_ships["shear_mag"] * np.cos(np.deg2rad(df_ships["shear_dir"]))
df_ships = df_ships.set_index("datetime")

#########################
# Interpolate shear to track times
#########################
def interpolate_shear(df_ships, interp_times):
    df_interp = pd.DataFrame(index=interp_times)
    for col in ["u_shear", "v_shear", "shear_mag"]:
        df_interp[col] = np.interp(
            interp_times.astype(int),
            df_ships.index.astype(int),
            df_ships[col]
        )
    return df_interp

df_ships_interp = interpolate_shear(df_ships, track_interp.index)

#########################
# Generate frames
#########################
for storm_time in track_interp.index:
    print(f"Processing {storm_time}...")

    # Storm position
    storm_lat = track_interp.loc[storm_time, "lat"]
    storm_lon = track_interp.loc[storm_time, "lon"]

    # Snap to MERGIR grid
    storm_lon_grid = ds_merg.lon.values[np.abs(ds_merg.lon.values - storm_lon).argmin()]
    storm_lat_grid = ds_merg.lat.values[np.abs(ds_merg.lat.values - storm_lat).argmin()]

    # Extract brightness temperature
    tb = ds_merg["Tb"].sel(time=storm_time, method="nearest").squeeze()

    # Subset region ±6°
    lat_min, lat_max = storm_lat_grid - 6, storm_lat_grid + 6
    lon_min, lon_max = storm_lon_grid - 6, storm_lon_grid + 6
    tb_sub = tb.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

    # Lightning ±15 min
    start = storm_time - pd.Timedelta(minutes=15)
    end = storm_time + pd.Timedelta(minutes=15)
    mask_time = (df_light_all["datetime"] >= start) & (df_light_all["datetime"] <= end)
    df_sub = df_light_all[
        mask_time &
        df_light_all["latitude"].between(lat_min, lat_max) &
        df_light_all["longitude"].between(lon_min, lon_max)
    ]
    print(f"Lightning strikes in ±15 min: {len(df_sub)}")

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8, 8))
    tb_sub.plot(cmap="turbo", vmin=180, vmax=300, add_colorbar=True, ax=ax)

    # Storm center + lightning
    ax.scatter(storm_lon_grid, storm_lat_grid, c="red", marker="x", s=100, zorder=5)
    ax.scatter(df_sub["longitude"], df_sub["latitude"], color="purple", s=4, alpha=0.6, zorder=4)

    # --- Radius circles (100 km and 350 km) ---
    for radius_km, color in zip([100, 350], ["deepskyblue", "orange"]):
        circ = Circle(
            (storm_lon_grid, storm_lat_grid),
            radius_km / 111.0,
            edgecolor=color,
            facecolor="none",
            linestyle="--",
            linewidth=1.4,
            alpha=0.9,
            zorder=5
        )
        ax.add_patch(circ)

    # --- RMW circle ---
    rmw_km = df_rmw_interp.loc[storm_time, 'RMW_km']
    circ_rmw = Circle(
        (storm_lon_grid, storm_lat_grid),
        rmw_km / 111.0,
        edgecolor='red',
        facecolor='none',
        linestyle='-',
        linewidth=2,
        alpha=0.8,
        zorder=5
    )
    ax.add_patch(circ_rmw)

    # --- Aspect ratio fix (physically correct for latitude) ---
    ax.set_aspect(1.0 / np.cos(np.deg2rad(storm_lat)))

    # --- Shear vector: normalize to fixed display length so magnitude
    #     doesn't affect arrow size — only direction is shown visually.
    #     Actual magnitude is shown in the legend and text label. ---
    shear = df_ships_interp.loc[storm_time]
    u, v, mag = shear["u_shear"], shear["v_shear"], shear["shear_mag"]

    display_length_deg = 1.5          # fixed arrow length in degrees
    vec_len = np.sqrt(u**2 + v**2)
    if vec_len > 0:
        u_norm = u / vec_len * display_length_deg
        v_norm = v / vec_len * display_length_deg
    else:
        u_norm, v_norm = 0.0, 0.0

    corner_lon = lon_min + 0.5
    corner_lat = lat_min + 0.5
    ax.quiver(
        corner_lon, corner_lat, u_norm, v_norm,
        angles="xy", scale_units="xy", scale=1,
        color="seagreen", width=0.005, headwidth=3, headlength=4,
        zorder=6
    )
    ax.text(
        corner_lon + 0.3, corner_lat + 0.2,
        f"{mag:.0f} kt shear", color="lime",
        fontsize=9, ha="left", va="bottom", weight="bold", alpha=0.8
    )

    # --- Intensity text ---
    stage = track_interp.loc[storm_time, "stage"]
    wind = track_interp.loc[storm_time, "wind"]
    ax.text(
        lon_max - 1.0, lat_min + 0.5,
        f"{stage.title()}\nWind: {wind:.0f} kt",
        fontsize=10, fontweight="bold",
        color="black", ha="right", va="bottom",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.6, boxstyle="round,pad=0.2")
    )

    

    # --- Legend (manual Line2D handles so circles render as lines) ---
    legend_elements = [
        Line2D([0], [0], marker='x', color='red', linestyle='None',
               markersize=8, label='Storm center'),
        Line2D([0], [0], marker='o', color='purple', linestyle='None',
               markersize=4, alpha=0.6, label='Lightning'),
        Line2D([0], [0], color='red', linewidth=2, linestyle='-',
               label=f'RMW ({rmw_km:.0f} km)'),
        Line2D([0], [0], color='deepskyblue', linewidth=1.4, linestyle='--',
               label='100 km radius'),
        Line2D([0], [0], color='orange', linewidth=1.4, linestyle='--',
               label='350 km radius'),
        Line2D([0], [0], color='seagreen', linewidth=2,
               label=f'Shear dir. ({mag:.0f} kt)'),
    ]
    ax.legend(
        handles=legend_elements,
        loc="upper right", fontsize=8,
        frameon=True, framealpha=0.7,
        facecolor="white", edgecolor="gray"
    )

    # --- Style ---
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"Hurricane Edouard - Brightness Temp + Lightning + SHIPS Shear\n{storm_time}")

    # --- Save frame ---
    outname = os.path.join(output_dir, f"edouard_overlay_{storm_time:%Y%m%d%H%M}.png")
    plt.savefig(outname, dpi=150, bbox_inches="tight")
    plt.close(fig)

#########################
# Combine frames into video
#########################
output_video = "edouard_overlay_20140911_20140918.mp4"
frame_rate = 6  # fps
subprocess.run([
    "ffmpeg", "-framerate", str(frame_rate),
    "-pattern_type", "glob", "-i", f"{output_dir}/*.png",
    "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-y",
    output_video
])
print(f"✅ Video saved as: {output_video}")